# 걷기 동영상 스캔 v4 A/B — v1 대비 (동영상1: 83권/안정 64)

**내용:** 동영상 3편(862-912 구간)을 1fps 프레임으로 쪼개 파이프라인 배치 → 프레임 투표 합산 =
"서가 한 줄 = 30초 산책" 첫 실증. 단 세트 사진 25장도 함께 처리.

**업로드:** `daelim_video_v4.zip` · GPU(T4) 런타임 · 셀1 후 **세션 다시 시작**

In [ ]:
# 1) 설치 — torch 제거(NCCL 충돌 방지) 후 최신 GPU paddle (>=3.3, 파인튜닝 모델 포맷 요구)
!pip uninstall -y -q torch torchvision torchaudio 2>/dev/null
!pip install -q paddlepaddle-gpu -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!pip install -q paddleocr
import paddle; print('paddle', paddle.__version__)
print('✅ 설치 완료 — [런타임 → 세션 다시 시작] 후 셀2부터!')

In [ ]:
# 2) 패키지 업로드 + 배치
from google.colab import files
up = files.upload()   # daelim_video_v4.zip
!unzip -oq daelim_video_v4.zip -d work/
%cd work
!ls videos/ sets/ | head

In [ ]:
# 3) 동영상 → 프레임 추출 (1fps)
import cv2, os
FPS_SAMPLE = 1.0
os.makedirs('frames', exist_ok=True)
for v in sorted(os.listdir('videos')):
    cap = cv2.VideoCapture(f'videos/{v}')
    fps = cap.get(cv2.CAP_PROP_FPS); step = int(round(fps / FPS_SAMPLE))
    i = n = 0
    while True:
        ok, fr = cap.read()
        if not ok: break
        if i % step == 0:
            cv2.imwrite(f'frames/{v.split(".")[0]}_f{i:05d}.jpg', fr, [cv2.IMWRITE_JPEG_QUALITY, 95]); n += 1
        i += 1
    cap.release(); print(v, '→', n, '프레임')

In [ ]:
# 4) 배치 실행 — 동영상 프레임(제목복구 생략=고속) + 세트 사진(전체 파이프라인)
import glob, subprocess, sys, time
def run(p, extra=[]):
    r = subprocess.run([sys.executable, '-u', 'daelim_closeup.py', p,
                        '--rec_dir', 'korean_lowres_v4_rec_infer', '--catalog', 'catalog_900.csv'] + extra,
                       capture_output=True, text=True)
    for ln in r.stdout.splitlines():
        if ln.startswith('[이중') or ln.startswith('[밴드'): print(' ', ln)
    if r.returncode != 0: print(r.stderr[-800:])
t0 = time.time()
fr_list = sorted(glob.glob('frames/*.jpg'))
for k, p in enumerate(fr_list):
    print(f'== 프레임 {k+1}/{len(fr_list)} {p} ({(time.time()-t0)/60:.0f}분 경과)')
    run(p, ['--no_title'])


In [ ]:
# 5) 동영상별 투표 합산 — 걷기 스캔 성적표 (v1 기준: 동영상1 83권/64 · 2 85/72 · 3 85/79)
import json, glob, re, csv
from collections import Counter
cat_status = {}
for r in csv.DictReader(open('catalog_900.csv', encoding='utf-8-sig')):
    cat_status[r['call_number'].strip()] = r['status']
vids = {}
for f in glob.glob('out_ondevice/*_f?????_*result.json'):
    m = re.search(r'(\d)_f(\d{5})', f)
    if m: vids.setdefault(m.group(1), []).append(f)
agg = {}
for v in sorted(vids):
    votes = Counter(); mis = Counter()
    for f in sorted(vids[v]):
        for r in json.load(open(f, encoding='utf-8')):
            if r['call']:
                votes[r['call']] += 1
                if r.get('mis'): mis[r['call']] += 1
    stable = {c: n for c, n in votes.items() if n >= 2}
    loaned = [c for c in stable if cat_status.get(c) and cat_status[c] not in ('비치자료', '')]
    agg[f'동영상{v}'] = {'frames': len(vids[v]), 'votes': dict(votes), 'stable': len(stable),
                       'mis': {c: n for c, n in mis.items() if n >= 2}, 'loaned_on_shelf': loaned}
    print(f'===== 동영상{v} ({len(vids[v])}프레임) =====')
    print(f'  고유 {len(votes)}권 · 2프레임 이상 안정 {len(stable)}권 · 5표 이상 {sum(1 for n in votes.values() if n >= 5)}권')
    print(f'  오배열(2표+): {agg[f"동영상{v}"]["mis"]} · 대출중 발견: {loaned}')
json.dump(agg, open('out_ondevice/aggregate_video.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
print('저장 완료')


In [ ]:
# 6) 결과 다운로드
!zip -q -r ../daelim_video_v4_results.zip out_ondevice
from google.colab import files
files.download('../daelim_video_v4_results.zip')